# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset includes mass spectrometry analysis of proteins from extracellular vesicles derived from stimulated human mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This is crucial to identify the components you'll interact with during analysis.

All entities are referenced by their `@id`. Let's print all record sets with their IDs, and list each field with the corresponding `@id` and type.

In [ ]:
# List available record sets and their fields, referencing all by their `@id`
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` fields from the overview above.

In this notebook, we extract all record sets for demonstration, but you can focus on a particular set if desired.

In [ ]:
# Extract data from each record set using the record set `@id`

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

print("Loading records for each record set by @id...")
for rs in dataset.record_sets:
    print(f" - Loading record set {rs.name} (id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
    else:
        print(f"   [No records found for {rs.name}]")

# For demonstration, select the first record set found
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    selected_rs_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set {selected_rs_id}:\n{dataframes[selected_rs_id].columns.tolist()}")
    dataframes[selected_rs_id].head()
else:
    print('No record sets with records found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All fields are referenced by their `@id` as defined in the dataset.

In [ ]:
# Select a record set and numeric field for EDA, referenced by @id
# For demonstration, pick the first available numeric field (Float/Integer/Number)
import numpy as np

selected_rs = None
numeric_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    # Find the first numeric field in this record set
    for field in rs.fields:
        # Consider 'Float', 'Integer', 'Number' as numeric
        if str(field.data_type).lower() in ["float", "integer", "number"]:
            selected_rs = rs
            numeric_field_id = field.id
            break
    # Optionally, find a non-numeric field for grouping
    if selected_rs:
        for field in selected_rs.fields:
            if str(field.data_type).lower() == "text":
                group_field_id = field.id
                break
        break

if selected_rs and numeric_field_id and selected_rs.id in dataframes:
    df = dataframes[selected_rs.id]
    colname = numeric_field_id
    print(f"Using record set: '{selected_rs.name}' (id: {selected_rs.id})")
    print(f"Numeric field: '{numeric_field_id}'")
    if group_field_id:
        print(f"Group field: '{group_field_id}'")
    # Ensure the column is numeric
    df[colname] = pd.to_numeric(df[colname], errors='coerce')
    # Remove NaNs for filtering
    df = df.dropna(subset=[colname])
    # Set a quantile-based threshold if there are large values
    threshold = df[colname].quantile(0.75)
    filtered_df = df[df[colname] > threshold]
    print(f"Filtered records with {colname} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{colname}_normalized"
    filtered_df[norm_col] = (filtered_df[colname] - filtered_df[colname].mean()) / filtered_df[colname].std()
    print(f"\nNormalized {colname} for filtered records:")
    print(filtered_df[[colname, norm_col]].head())

    # Group by a group field if found
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[colname].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {colname}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field or records found. Please check record set availability.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and show a boxplot grouped by the chosen field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No suitable filtered DataFrame for visualization.')

## 6. Conclusion
In this notebook, we loaded a FAIR² dataset using `mlcroissant`, explored record sets and their fields by `@id`, extracted records for analysis, performed EDA, and visualized key numeric distributions. This process illustrates how `mlcroissant` enables transparent FAIR data exploration using schema-based identifiers for robust and reproducible workflows.